In [1]:
!pip install sentence-transformers scikit-learn numpy

In [7]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

def preprocess(text):
    return [line.strip().lower() for line in text.split("\n") if line.strip()]

def is_number(line):
    return bool(re.fullmatch(r"\d+(\.\d+)?", line))

def extract_price(line):
    match = re.search(r"\d+(\.\d+)?", line)
    return float(match.group()) if match else None

def merge_lines(lines):
    merged = []
    buffer = ""

    for line in lines:
        if is_number(line):
            if buffer:
                merged.append(buffer + " " + line)
                buffer = ""
        else:
            if buffer:
                buffer += " " + line
            else:
                buffer = line

    return merged


def process_receipt(text):
    lines = preprocess(text)

    taxes = []
    total = None
    clean_lines = []

    for line in lines:
        if "total" in line:
            total = extract_price(line)
        elif "gst" in line or "tax" in line:
            val = extract_price(line)
            if val:
                taxes.append(val)
        else:
            clean_lines.append(line)


    merged_lines = merge_lines(clean_lines)

    items = []

    for line in merged_lines:
        match = re.search(r"(.+?)\s+(\d+(\.\d+)?)$", line)
        if match:
            items.append({
                "name": match.group(1),
                "price": float(match.group(2))
            })

    return {
        "items": items,
        "tax": sum(taxes),
        "total": total
    }



text = """Paneer
Tikka
250
Butter
Naan
40
Masala Dosa
120
CGST 15
SGST 15
Total 440"""

result = process_receipt(text)

print(" Correct Output:\n")
print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Correct Output:

{'items': [{'name': 'paneer tikka', 'price': 250.0}, {'name': 'butter naan', 'price': 40.0}, {'name': 'masala dosa', 'price': 120.0}], 'tax': 30.0, 'total': 440.0}
